# 🔭 Entraînement BYOL — Features Self-Supervised pour Galaxies COSMOS
Auteur : Tojolalaina Cédric RANDRIAMAROTIA\
Date de la première édition : 13/05/2026\
Outils : Antigravity, Claude Sonnet 4.6 


Ce notebook entraîne un modèle BYOL (Bootstrap Your Own Latent) sur des images
de galaxies multi-bandes du survey COSMOS (HSC-CLAUDS) pour extraire des
représentations (features) self-supervised, sans utiliser de labels de redshift.

**Références :**
- Grill et al. (2020) — BYOL : Bootstrap Your Own Latent
- Treyer, Ait Ouahmed et al. (2024-2026) — CNN multi-modal pour photo-z


### 🚀 From Scratch & Opposite Cropping
- **PairedGalaxyAugmentation** : Cadrages diagnoalement opposés de 48x48 pour forcer l'apprentissage sur l'overlap de 32x32 au centre.
- **From Scratch** : `PRETRAINED = False`, `LR = 1e-3`.
- **Optimisations** : `BATCH_SIZE = 256`, 12000 points dans le t-SNE final, avec toutes les variantes multi-paramètres.

## Section 0 — Configuration & Environnement


In [ ]:
# === Installation des dépendances (Colab) ===
# Décommenter les lignes suivantes si exécuté sur Google Colab :
# !pip install -q albumentations


In [ ]:
import os
import copy
import time
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision.models as models

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# Reproductibilité
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# === Configuration centralisée ===


CONFIG = {
    # --- Données ---
    "IMAGE_SIZE": 64,            # Taille des images (pas de resize nécessaire)
    "N_BANDS": 6,                # Bandes retenues : u, g, r, i, z, y
    "BAND_INDICES": [0, 1, 2, 3, 4, 5],  # Indices dans le cube 9-bandes
    "BAND_NAMES": ["u", "g", "r", "i", "z", "y"],
    # Modalités (groupement par type de bande, inspiré de Treyer)
    "MODALITIES": {
        "optical": [0, 1, 2],    # u, g, r — bandes optiques bleues
        "red_nir": [3, 4, 5],    # i, z, y — bandes rouges/proche-IR
    },

    # --- Architecture BYOL ---
    "FEAT_DIM": 512,             # Dimension de sortie de l'encodeur (ResNet18)
    "HIDDEN_DIM": 4096,          # Dimension cachée des MLP (projector/predictor)
    "PROJ_DIM": 256,             # Dimension de projection finale
    "PRETRAINED": False,        

    # --- Entraînement ---
    "BATCH_SIZE": 256,           # Optimisé pour T4 15GB avec mixed precision
    "NUM_EPOCHS": 100,           # 100 epochs (résultats visibles dès ~20 epochs)
    "LR": 1e-3,                  # Learning rate (standard BYOL)
    "WEIGHT_DECAY": 1e-4,        # Régularisation L2
    "EMA_MOMENTUM": 0.996,       # Momentum pour la mise à jour EMA du target
    "NUM_WORKERS": 2,            # Workers pour le DataLoader (Colab)

    # --- Augmentations ---
    "CROP_SIZE": 48,    # Échelle du random crop
    "NOISE_STD_RANGE": (0.01, 0.05),  # Gamme du bruit gaussien

    # --- Chemins (adapter selon votre Google Drive) ---
    "DATA_DIR": "Data/",         # Dossier contenant les fichiers .npz
    "SAVE_DIR": "checkpoints/",  # Dossier de sauvegarde des modèles
    "CHECKPOINT_EVERY": 20,      # Sauvegarde tous les N epochs
}

# Vérification GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device : {device}")
if torch.cuda.is_available():
    print(f"🔥 GPU : {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

os.makedirs(CONFIG["SAVE_DIR"], exist_ok=True)


## Section 1 — Chargement et préparation des données

Les données COSMOS sont stockées dans des fichiers `.npz` contenant :
- `cube` : images multi-bandes (N, 64, 64, 9) — 9 bandes photométriques
- `info` : métadonnées structurées (ID, coordonnées, magnitudes, ZPHOT, ZSPEC...)
- `flag` : indicateurs de qualité par bande (N, 9)

On ne retient que 6 bandes (u, g, r, i, z, y), en excluant J, H, Ks.
**Pas d'imputation** des valeurs manquantes (NaN) : c'est le contexte de
mismatch entre domaines que l'on souhaite étudier.


In [ ]:
def load_cosmos_data(data_dir, band_indices):
    """
    Charge tous les fichiers .npz du répertoire et retourne les cubes
    (images) et les métadonnées, en ne gardant que les bandes sélectionnées.

    Returns:
        cubes_train : np.ndarray (N_train, C, H, W) — images pour l'entraînement BYOL
        cubes_eval  : np.ndarray (N_eval, C, H, W) — images avec ZSPEC pour évaluation
        info_train  : structured array — métadonnées entraînement
        info_eval   : structured array — métadonnées évaluation
        zphot_train : np.ndarray — redshifts photométriques (entraînement)
        zspec_eval  : np.ndarray — redshifts spectroscopiques (évaluation)
        zphot_eval  : np.ndarray — redshifts photométriques (évaluation)
        survey_labels : list — 'D' ou 'UD' pour chaque galaxie d'évaluation
    """
    # Fichiers de données
    files_train = [
        "COSMOS_v11_uijk_0001_photo_D.npz",     # ~12K galaxies Deep
        "COSMOS_v11_uijk_0213_photo_UD.npz",     # 443 galaxies Ultra-Deep
    ]
    files_eval = [
        "COSMOS_v11_uijk_0020_spec_D.npz",       # 15 galaxies Deep (avec ZSPEC)
        "COSMOS_v11_uijk_0073_spec_UD.npz",      # 12 galaxies Ultra-Deep (avec ZSPEC)
    ]

    cubes_list_train, info_list_train = [], []
    cubes_list_eval, info_list_eval = [], []
    survey_labels = []

    # --- Chargement des données d'entraînement (photo, sans ZSPEC) ---
    for fname in files_train:
        fpath = os.path.join(data_dir, fname)
        if not os.path.exists(fpath):
            print(f"⚠️  Fichier non trouvé : {fpath}")
            continue
        print(f"📂 Chargement de {fname}...", end=" ")
        d = np.load(fpath, allow_pickle=True)
        cube = d['cube'][:, :, :, band_indices]  # (N, 64, 64, 6) — sélection des bandes
        cube = np.transpose(cube, (0, 3, 1, 2))  # (N, 6, 64, 64) — channels first
        cubes_list_train.append(cube)
        info_list_train.append(d['info'])
        print(f"✅ {len(d['info'])} galaxies")
        d.close()

    # --- Chargement des données d'évaluation (spec, avec ZSPEC) ---
    for fname in files_eval:
        fpath = os.path.join(data_dir, fname)
        if not os.path.exists(fpath):
            print(f"⚠️  Fichier non trouvé : {fpath}")
            continue
        print(f"📂 Chargement de {fname}...", end=" ")
        d = np.load(fpath, allow_pickle=True)
        cube = d['cube'][:, :, :, band_indices]
        cube = np.transpose(cube, (0, 3, 1, 2))
        cubes_list_eval.append(cube)
        info_list_eval.append(d['info'])
        # Identifier le survey (D ou UD) pour la visualisation
        label = "UD" if "UD" in fname else "D"
        survey_labels.extend([label] * len(d['info']))
        print(f"✅ {len(d['info'])} galaxies ({label})")
        d.close()

    # Concaténation
    cubes_train = np.concatenate(cubes_list_train, axis=0).astype(np.float32)
    cubes_eval = np.concatenate(cubes_list_eval, axis=0).astype(np.float32)

    # Récupération des infos
    info_train = np.concatenate(info_list_train)
    info_eval = np.concatenate(info_list_eval)
    zphot_train = info_train['ZPHOT'].astype(np.float32)
    zspec_eval = info_eval['ZSPEC'].astype(np.float32)
    zphot_eval = info_eval['ZPHOT'].astype(np.float32)

    # Gestion des NaN : on remplace par 0 (= pas de signal)
    # Ce n'est PAS de l'imputation — c'est un remplacement technique pour
    # que le réseau puisse fonctionner. Les données manquantes restent "absentes".
    nan_count = np.isnan(cubes_train).sum()
    if nan_count > 0:
        print(f"⚠️  {nan_count} valeurs NaN remplacées par 0 dans les cubes d'entraînement")
        cubes_train = np.nan_to_num(cubes_train, nan=0.0)
    nan_count_eval = np.isnan(cubes_eval).sum()
    if nan_count_eval > 0:
        print(f"⚠️  {nan_count_eval} valeurs NaN remplacées par 0 dans les cubes d'évaluation")
        cubes_eval = np.nan_to_num(cubes_eval, nan=0.0)

    print(f"\n📊 Résumé des données :")
    print(f"   Entraînement BYOL : {len(cubes_train)} galaxies ({cubes_train.shape})")
    print(f"   Évaluation (ZSPEC): {len(cubes_eval)} galaxies ({cubes_eval.shape})")
    print(f"   ZPHOT range (train): [{zphot_train.min():.3f}, {zphot_train.max():.3f}]")
    print(f"   ZSPEC range (eval) : [{zspec_eval.min():.3f}, {zspec_eval.max():.3f}]")

    return (cubes_train, cubes_eval, info_train, info_eval,
            zphot_train, zspec_eval, zphot_eval, survey_labels)


In [ ]:
# === Normalisation par bande (Option 1 — standardisation) ===
def compute_band_stats(cubes):
    """Calcule la moyenne et l'écart-type par bande sur tout le dataset."""
    # cubes: (N, C, H, W)
    mean = np.nanmean(cubes, axis=(0, 2, 3))  # (C,)
    std = np.nanstd(cubes, axis=(0, 2, 3))    # (C,)
    std[std < 1e-8] = 1.0  # Éviter la division par zéro
    return mean, std


def normalize_cubes(cubes, mean, std):
    """Normalise les cubes par bande (z-score)."""
    # Broadcasting : mean/std (C,) -> (1, C, 1, 1)
    mean = mean.reshape(1, -1, 1, 1)
    std = std.reshape(1, -1, 1, 1)
    return (cubes - mean) / std


## Section 2 — Augmentations pour BYOL


- Le random crop enseigne l'invariance spatiale (la position dans l'image
  n'est pas informative pour la morphologie de la galaxie)
- Le bruit gaussien enseigne la robustesse au bruit instrumental
  (omniprésent en imagerie astronomique)

**Note sur les augmentations absentes** :
- Les flips H/V et rotations 90° sont physiquement gratuits (les galaxies
  n'ont pas d'orientation préférentielle) et amélioreraient les features,
  mais ne sont pas inclus selon la consigne du commanditaire.
- Le ColorJitter est exclu car les bandes astronomiques ont une
  signification physique (distribution spectrale d'énergie).


In [ ]:
class PairedGalaxyAugmentation:
    """
    Opposite Cropping: Génère deux vues couplées recadrées de manière diamétralement opposée.
    L'overlap est garanti et correspond exactement au centre de l'image (où se trouve la galaxie).
    """
    def __init__(self, crop_size=48, image_size=64, noise_std_range=(0.01, 0.05)):
        self.crop_size = crop_size
        self.image_size = image_size
        self.noise_std_range = noise_std_range

    def __call__(self, x):
        # x : torch.Tensor (C, H, W)
        C, H, W = x.shape
        max_offset = H - self.crop_size
        
        # Vue 1 : décalage aléatoire
        top1 = random.randint(0, max_offset)
        left1 = random.randint(0, max_offset)
        
        # Vue 2 : décalage mathématiquement opposé
        top2 = max_offset - top1
        left2 = max_offset - left1
        
        v1 = x[:, top1:top1+self.crop_size, left1:left1+self.crop_size]
        v2 = x[:, top2:top2+self.crop_size, left2:left2+self.crop_size]
        
        # Resize bilinéaire vers image_size
        v1 = F.interpolate(v1.unsqueeze(0), size=(self.image_size, self.image_size), mode='bilinear', align_corners=False).squeeze(0)
        v2 = F.interpolate(v2.unsqueeze(0), size=(self.image_size, self.image_size), mode='bilinear', align_corners=False).squeeze(0)
        
        # Bruit gaussien indépendant
        std1 = random.uniform(*self.noise_std_range)
        std2 = random.uniform(*self.noise_std_range)
        v1 = v1 + torch.randn_like(v1) * std1
        v2 = v2 + torch.randn_like(v2) * std2
        
        return v1, v2

class BYOLGalaxyDataset(Dataset):
    """Dataset BYOL utilisant PairedGalaxyAugmentation pour retourner les deux vues opposées."""
    def __init__(self, cubes, paired_transform):
        self.cubes = torch.from_numpy(cubes)
        self.transform = paired_transform
        print(f"✅ BYOL Dataset : {len(self.cubes)} galaxies chargées")

    def __len__(self):
        return len(self.cubes)

    def __getitem__(self, idx):
        image = self.cubes[idx]
        view1, view2 = self.transform(image)
        return view1, view2


## Section 3 — Architecture du modèle BYOL

### Encodeur avec traitement par modalité
Inspiré de l'architecture de Treyer (Parellel_Inception), l'encodeur
traite les bandes optiques (u, g, r) et rouges/NIR (i, z, y) dans des
branches parallèles avant de les fusionner et de les passer dans un
backbone ResNet18 (avec poids ImageNet pré-entraînés pour les couches 1-4).

### Structure BYOL
- **Online network** : Encodeur + Projecteur + Prédicteur (entraîné par gradient)
- **Target network** : Encodeur + Projecteur (mis à jour par EMA, τ=0.996)


In [ ]:
class GalaxyEncoder(nn.Module):
    """
    Encodeur de galaxies avec traitement par modalité.

    Architecture :
        1. Deux branches parallèles (optical: u,g,r et red/NIR: i,z,y)
           inspirées du before_inception_block de Treyer (Conv 5×5, PReLU/Tanh)
        2. Fusion par concaténation → 64 canaux
        3. Backbone ResNet18 (layers 1-4, poids ImageNet pré-entraînés)
        4. Global Average Pooling → vecteur 512-dim

    Entrée : (B, 6, 64, 64) — 6 bandes
    Sortie : (B, 512) — features
    """
    def __init__(self, pretrained=True):
        super().__init__()

        # === Branches par modalité ===
        # Chaque modalité : Conv 5×5 → BN → PReLU → Conv 5×5 → BN → Tanh
        self.optical_stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm2d(32),
            nn.PReLU(),
            nn.Conv2d(32, 32, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm2d(32),
            nn.Tanh()
        )
        self.red_nir_stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm2d(32),
            nn.PReLU(),
            nn.Conv2d(32, 32, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm2d(32),
            nn.Tanh()
        )

        # === Backbone ResNet18 ===
        resnet = models.resnet18(
            weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        )

        # Couche de fusion : 64 canaux (32+32) → 64 canaux
        # Remplace le conv1 original (3→64) par un nouveau (64→64)
        self.conv1 = nn.Conv2d(64, 64, kernel_size=7, stride=2, padding=3, bias=False)
        nn.init.kaiming_normal_(self.conv1.weight, mode='fan_out', nonlinearity='relu')

        # Couches du ResNet18 avec poids ImageNet (layers 1-4)
        self.bn1 = resnet.bn1        # Pré-entraîné
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1  # 64 → 64   (pré-entraîné)
        self.layer2 = resnet.layer2  # 64 → 128  (pré-entraîné)
        self.layer3 = resnet.layer3  # 128 → 256 (pré-entraîné)
        self.layer4 = resnet.layer4  # 256 → 512 (pré-entraîné)

        self.feature_dim = 512

    def forward(self, x):
        # x: (B, 6, 64, 64)
        optical = self.optical_stem(x[:, :3])   # u,g,r → (B, 32, 64, 64)
        red_nir = self.red_nir_stem(x[:, 3:])   # i,z,y → (B, 32, 64, 64)

        x = torch.cat([optical, red_nir], dim=1)  # (B, 64, 64, 64)

        # Backbone ResNet18
        x = self.conv1(x)     # (B, 64, 32, 32)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)   # (B, 64, 16, 16)
        x = self.layer1(x)    # (B, 64, 16, 16)
        x = self.layer2(x)    # (B, 128, 8, 8)
        x = self.layer3(x)    # (B, 256, 4, 4)
        x = self.layer4(x)    # (B, 512, 2, 2)

        # Global Average Pooling
        x = x.mean(dim=[2, 3])  # (B, 512)
        return x


In [ ]:
class MLPHead(nn.Module):
    """
    MLP à 2 couches avec BatchNorm — utilisé comme Projecteur et Prédicteur.
    Architecture standard BYOL : Linear → BN → ReLU → Linear
    """
    def __init__(self, in_dim, hidden_dim=4096, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
class BYOL(nn.Module):
    """
    Bootstrap Your Own Latent (BYOL) — Grill et al., NeurIPS 2020

    Architecture :
    - Online network  : Encodeur fθ + Projecteur gθ + Prédicteur qθ
      → Entraîné par descente de gradient
    - Target network  : Encodeur fξ + Projecteur gξ
      → Mis à jour par Exponential Moving Average (EMA) : ξ ← τξ + (1-τ)θ

    La loss minimise la distance cosinus entre la prédiction online
    et la projection target, de manière symétrique (les deux vues
    servent alternativement d'online et de target).
    """
    def __init__(self, pretrained=True, feat_dim=512, hidden_dim=4096, proj_dim=256):
        super().__init__()

        # --- Online network (entraîné par gradient) ---
        self.online_encoder = GalaxyEncoder(pretrained=pretrained)
        self.online_projector = MLPHead(feat_dim, hidden_dim, proj_dim)
        self.online_predictor = MLPHead(proj_dim, hidden_dim, proj_dim)

        # --- Target network (copie EMA, pas de gradient) ---
        self.target_encoder = copy.deepcopy(self.online_encoder)
        self.target_projector = MLPHead(feat_dim, hidden_dim, proj_dim)

        # Désactiver les gradients pour le target
        for param in self.target_encoder.parameters():
            param.requires_grad = False
        for param in self.target_projector.parameters():
            param.requires_grad = False

        # Copie exacte des poids au début (momentum=0)
        self._update_target(momentum=0.0)

    @torch.no_grad()
    def _update_target(self, momentum=0.996):
        """Mise à jour EMA du target network : ξ ← τξ + (1-τ)θ"""
        for op, tp in zip(self.online_encoder.parameters(),
                          self.target_encoder.parameters()):
            tp.data = momentum * tp.data + (1 - momentum) * op.data

        for op, tp in zip(self.online_projector.parameters(),
                          self.target_projector.parameters()):
            tp.data = momentum * tp.data + (1 - momentum) * op.data

    def forward(self, view1, view2):
        """
        Forward pass BYOL symétrique.

        Entrées : view1, view2 — deux vues augmentées (B, C, H, W)
        Sorties : pred1, pred2 — prédictions online
                  target_proj1, target_proj2 — projections target (détachées)
        """
        # Online : vue 1 → encoder → projector → predictor
        feat1 = self.online_encoder(view1)
        proj1 = self.online_projector(feat1)
        pred1 = self.online_predictor(proj1)

        # Online : vue 2
        feat2 = self.online_encoder(view2)
        proj2 = self.online_projector(feat2)
        pred2 = self.online_predictor(proj2)

        # Target : pas de gradient (stop gradient)
        with torch.no_grad():
            target_feat1 = self.target_encoder(view1)
            target_proj1 = self.target_projector(target_feat1)
            target_feat2 = self.target_encoder(view2)
            target_proj2 = self.target_projector(target_feat2)

        return pred1, pred2, target_proj1.detach(), target_proj2.detach()


In [ ]:
def byol_loss(pred1, pred2, target_proj1, target_proj2):
    """
    Loss BYOL symétrique basée sur la similarité cosinus.

    L = 2 - 2 * cos(pred1, target_proj2)  (vue 1 prédit vue 2)
      + 2 - 2 * cos(pred2, target_proj1)  (vue 2 prédit vue 1)

    Normalisée par L2 avant le calcul du produit scalaire.
    """
    pred1 = F.normalize(pred1, dim=-1)
    pred2 = F.normalize(pred2, dim=-1)
    target_proj1 = F.normalize(target_proj1, dim=-1)
    target_proj2 = F.normalize(target_proj2, dim=-1)

    loss1 = 2 - 2 * (pred1 * target_proj2).sum(dim=-1).mean()
    loss2 = 2 - 2 * (pred2 * target_proj1).sum(dim=-1).mean()

    return (loss1 + loss2) / 2


## Section 4 — Boucle d'entraînement BYOL

1. Pour chaque batch, deux vues augmentées sont créées
2. Le réseau online prédit les projections du réseau target
3. La loss (cosine similarity) est rétropropagée uniquement dans le online
4. Le réseau target est mis à jour par EMA (τ=0.996)




In [ ]:
def train_byol(model, dataloader, config, device):
    """
    Entraînement complet du modèle BYOL.

    Returns:
        history : dict avec les courbes de loss et learning rate
    """
    # Optimiseur : uniquement les paramètres du online network
    optimizer = torch.optim.Adam(
        list(model.online_encoder.parameters()) +
        list(model.online_projector.parameters()) +
        list(model.online_predictor.parameters()),
        lr=config["LR"],
        weight_decay=config["WEIGHT_DECAY"]
    )

    # Scheduler : Cosine Annealing (décroissance progressive du LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config["NUM_EPOCHS"]
    )

    # Mixed precision pour économiser la VRAM sur T4
    scaler = GradScaler()

    # Historique
    history = {'loss': [], 'lr': []}

    print(f"\n⏳ Début du pré-entraînement BYOL")
    print(f"   {len(dataloader.dataset)} galaxies | {len(dataloader)} batches/epoch")
    print(f"   {config['NUM_EPOCHS']} epochs | LR={config['LR']} | Momentum={config['EMA_MOMENTUM']}")
    print(f"   Mixed precision : {'✅' if torch.cuda.is_available() else '❌'}")
    print("=" * 60)

    start_time = time.time()

    for epoch in range(config["NUM_EPOCHS"]):
        model.train()
        epoch_loss = 0.0

        for view1, view2 in dataloader:
            view1 = view1.to(device, non_blocking=True)
            view2 = view2.to(device, non_blocking=True)

            optimizer.zero_grad()

            # Forward avec mixed precision
            with autocast():
                pred1, pred2, tp1, tp2 = model(view1, view2)
                loss = byol_loss(pred1, pred2, tp1, tp2)

            # Backward (uniquement online network)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            # Mise à jour EMA du target network
            model._update_target(momentum=config["EMA_MOMENTUM"])

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(dataloader)
        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step()

        history['loss'].append(avg_loss)
        history['lr'].append(current_lr)

        # Affichage régulier
        if (epoch + 1) % 10 == 0 or epoch == 0:
            elapsed = time.time() - start_time
            print(f"Epoch [{epoch+1:3d}/{config['NUM_EPOCHS']}] | "
                  f"Loss: {avg_loss:.4f} | LR: {current_lr:.6f} | "
                  f"Temps: {elapsed/60:.1f} min")

        # Checkpoint
        if (epoch + 1) % config["CHECKPOINT_EVERY"] == 0:
            ckpt_path = os.path.join(
                config["SAVE_DIR"],
                f"byol_epoch_{epoch+1}.pth"
            )
            torch.save({
                'epoch': epoch + 1,
                'encoder_state_dict': model.online_encoder.state_dict(),
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_loss,
            }, ckpt_path)
            print(f"   💾 Checkpoint sauvegardé : {ckpt_path}")

        # Détection de collapse (loss constante = features triviales)
        if epoch > 20 and len(history['loss']) > 20:
            recent = history['loss'][-20:]
            if max(recent) - min(recent) < 1e-5:
                print("⚠️  ATTENTION : Loss stagnante → possible collapse !")
                print("   Vérifiez les augmentations ou le learning rate.")

    total_time = time.time() - start_time
    print("=" * 60)
    print(f"🏁 Pré-entraînement terminé en {total_time/60:.1f} minutes")
    print(f"   Loss finale : {history['loss'][-1]:.4f}")

    # Sauvegarde finale de l'encodeur
    final_path = os.path.join(config["SAVE_DIR"], "byol_encoder_final.pth")
    torch.save(model.online_encoder.state_dict(), final_path)
    print(f"✅ Encodeur sauvegardé : {final_path}")

    return history


## Section 5 — Extraction des features

Après l'entraînement, on utilise l'encodeur online (sans projector/predictor)
pour extraire les features 512-dim de toutes les galaxies.
Ces features constituent la représentation self-supervised apprise par BYOL.


In [ ]:
@torch.no_grad()
def extract_features(encoder, cubes, device, batch_size=256):
    """
    Extrait les features 512-dim pour un ensemble de cubes de galaxies.

    Args:
        encoder : GalaxyEncoder entraîné
        cubes : np.ndarray (N, C, H, W) — images normalisées
        device : torch.device

    Returns:
        features : np.ndarray (N, 512) — features extraites
    """
    encoder.eval()
    dataset = torch.utils.data.TensorDataset(torch.from_numpy(cubes))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    all_features = []
    for (batch,) in loader:
        batch = batch.to(device)
        feat = encoder(batch)  # (B, 512) — GAP déjà appliqué dans l'encoder
        all_features.append(feat.cpu().numpy())

    return np.concatenate(all_features, axis=0)


## Section 6 — Évaluation des représentations apprises

### 6.1 — Visualisation t-SNE
Projection des features 512-dim en 2D via t-SNE. Un bon encodeur
produira un espace où les galaxies de redshifts similaires sont
regroupées (gradient de couleur continu).

### 6.2 — Similarité cosinus intra/inter-classe
Mesure quantitative : les galaxies de même bin de redshift doivent
avoir une similarité cosinus élevée entre elles (intra-classe),
et faible avec les galaxies de bins différents (inter-classe).


In [ ]:
def plot_training_curves(history, save_path="training_curves.png"):
    """Trace les courbes de loss et learning rate pendant l'entraînement."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Loss
    ax1.plot(history['loss'], color='#534AB7', linewidth=2)
    ax1.set_title('Pré-entraînement BYOL — Loss', fontweight='bold', fontsize=13)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('BYOL Loss (cosine distance)')
    ax1.grid(True, alpha=0.2)

    # Learning Rate
    ax2.plot(history['lr'], color='#E07A5F', linewidth=2)
    ax2.set_title('Learning Rate (Cosine Annealing)', fontweight='bold', fontsize=13)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('LR')
    ax2.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"✅ Courbes sauvegardées : {save_path}")
    plt.show()


def plot_tsne(features, redshifts, title="t-SNE features colorées par Z",
              survey_labels=None, save_path="tsne_byol.png"):
    """
    Visualisation t-SNE des features, colorées par redshift.

    Args:
        features : np.ndarray (N, 512) — features extraites
        redshifts : np.ndarray (N,) — valeurs de redshift (ZPHOT ou ZSPEC)
        survey_labels : list (N,) — 'D' ou 'UD' pour chaque galaxie (optionnel)
    """
    print("🔄 Calcul du t-SNE (peut prendre quelques minutes)...")

    # Standardisation avant t-SNE (recommandé)
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    # t-SNE
    tsne = TSNE(n_components=2, perplexity=min(30, len(features)-1),
                random_state=SEED, n_iter=1000)
    embedding = tsne.fit_transform(features_scaled)

    # Visualisation
    fig, ax = plt.subplots(figsize=(10, 8))

    if survey_labels is not None:
        # Marqueurs différents pour D et UD
        for label, marker in [("D", "o"), ("UD", "s")]:
            mask = np.array(survey_labels) == label
            if mask.any():
                sc = ax.scatter(embedding[mask, 0], embedding[mask, 1],
                               c=redshifts[mask], cmap='viridis',
                               s=20, alpha=0.7, marker=marker, label=label,
                               vmin=redshifts.min(), vmax=redshifts.max())
    else:
        sc = ax.scatter(embedding[:, 0], embedding[:, 1],
                        c=redshifts, cmap='viridis',
                        s=10, alpha=0.6)

    cbar = plt.colorbar(sc, ax=ax, label='Redshift (z)')
    ax.set_xlabel('Dimension 1', fontsize=12)
    ax.set_ylabel('Dimension 2', fontsize=12)
    ax.set_title(title, fontweight='bold', fontsize=14)
    if survey_labels is not None:
        ax.legend(loc='upper right', fontsize=10)
    ax.grid(True, alpha=0.1)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"✅ t-SNE sauvegardé : {save_path}")
    plt.show()


def compute_cosine_similarity_analysis(features, redshifts, n_bins=10):
    """
    Analyse de la similarité cosinus intra/inter-classe.

    Les galaxies sont regroupées en bins de redshift. Pour chaque paire de bins,
    on calcule la similarité cosinus moyenne. Un bon encodeur montre :
    - Similarité intra-classe élevée (diagonale de la matrice)
    - Similarité inter-classe faible (hors diagonale)
    - Décroissance de la similarité avec la distance en redshift

    Args:
        features : np.ndarray (N, D) — features normalisées
        redshifts : np.ndarray (N,) — valeurs de redshift
        n_bins : int — nombre de bins de redshift

    Returns:
        sim_matrix : np.ndarray (n_bins, n_bins) — matrice de similarité
        bin_edges : np.ndarray — bornes des bins
    """
    print(f"🔄 Analyse de similarité cosinus ({n_bins} bins de redshift)...")

    # Normalisation L2 des features
    norms = np.linalg.norm(features, axis=1, keepdims=True)
    norms[norms < 1e-8] = 1.0
    features_norm = features / norms

    # Discrétisation en bins de redshift
    bin_edges = np.linspace(redshifts.min(), redshifts.max(), n_bins + 1)
    bin_labels = np.digitize(redshifts, bin_edges) - 1
    bin_labels = np.clip(bin_labels, 0, n_bins - 1)

    # Matrice de similarité cosinus moyenne par paire de bins
    sim_matrix = np.zeros((n_bins, n_bins))
    counts_matrix = np.zeros((n_bins, n_bins))

    for i in range(n_bins):
        mask_i = bin_labels == i
        if not mask_i.any():
            continue
        feats_i = features_norm[mask_i]

        for j in range(i, n_bins):
            mask_j = bin_labels == j
            if not mask_j.any():
                continue
            feats_j = features_norm[mask_j]

            # Similarité cosinus moyenne entre les groupes
            sim = feats_i @ feats_j.T  # (ni, nj)
            if i == j:
                # Intra-classe : exclure la diagonale (auto-similarité = 1)
                np.fill_diagonal(sim, 0)
                n_pairs = sim.shape[0] * (sim.shape[0] - 1) if sim.shape[0] > 1 else 1
                sim_matrix[i, j] = sim.sum() / n_pairs
            else:
                sim_matrix[i, j] = sim.mean()
                sim_matrix[j, i] = sim_matrix[i, j]  # Symétrie

            counts_matrix[i, j] = mask_i.sum()
            counts_matrix[j, i] = mask_j.sum()

    return sim_matrix, bin_edges


def plot_cosine_similarity(sim_matrix, bin_edges, save_path="cosine_similarity.png"):
    """Visualise la matrice de similarité cosinus par bin de redshift."""
    fig, ax = plt.subplots(figsize=(8, 7))

    n_bins = len(bin_edges) - 1
    bin_centers = [(bin_edges[i] + bin_edges[i+1]) / 2 for i in range(n_bins)]
    bin_labels = [f"{c:.2f}" for c in bin_centers]

    im = ax.imshow(sim_matrix, cmap='RdYlBu_r', aspect='auto',
                   vmin=sim_matrix[sim_matrix > 0].min() if (sim_matrix > 0).any() else 0,
                   vmax=sim_matrix.max())

    ax.set_xticks(range(n_bins))
    ax.set_yticks(range(n_bins))
    ax.set_xticklabels(bin_labels, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(bin_labels, fontsize=8)
    ax.set_xlabel('Bin de Redshift (z)', fontsize=12)
    ax.set_ylabel('Bin de Redshift (z)', fontsize=12)
    ax.set_title('Similarité cosinus moyenne par bin de Redshift',
                 fontweight='bold', fontsize=13)

    plt.colorbar(im, ax=ax, label='Similarité cosinus')

    # Annoter les valeurs sur la diagonale
    for i in range(n_bins):
        if sim_matrix[i, i] > 0:
            ax.text(i, i, f"{sim_matrix[i,i]:.3f}",
                    ha='center', va='center', fontsize=7,
                    color='white' if sim_matrix[i,i] > 0.5 else 'black')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"✅ Matrice de similarité sauvegardée : {save_path}")
    plt.show()

    # Résumé quantitatif
    diag = np.diag(sim_matrix)
    diag_valid = diag[diag > 0]
    off_diag = sim_matrix[np.triu_indices(n_bins, k=1)]
    off_diag_valid = off_diag[off_diag > 0]

    print(f"\n📊 Résumé de la similarité cosinus :")
    if len(diag_valid) > 0:
        print(f"   Intra-classe (diagonale) : {diag_valid.mean():.4f} ± {diag_valid.std():.4f}")
    if len(off_diag_valid) > 0:
        print(f"   Inter-classe (hors diag) : {off_diag_valid.mean():.4f} ± {off_diag_valid.std():.4f}")
    if len(diag_valid) > 0 and len(off_diag_valid) > 0:
        ratio = diag_valid.mean() / (off_diag_valid.mean() + 1e-8)
        print(f"   Ratio intra/inter        : {ratio:.2f}")
        print(f"   → {'✅ Bon' if ratio > 1.2 else '⚠️ À améliorer'} "
              f"(ratio > 1.2 indique des features discriminantes)")


## Section 7 — Exécution principale & Rapport


1. Chargement des données
2. Normalisation par bande
3. Création du dataset et des augmentations
4. Entraînement BYOL
5. Extraction des features
6. Évaluation (t-SNE + similarité cosinus)
7. Rapport final


In [ ]:
def main():
    """Pipeline complet d'entraînement et d'évaluation BYOL."""

    print("=" * 60)
    print("🔭 BYOL — Pré-entraînement Self-Supervised pour Galaxies COSMOS")
    print("=" * 60)

    # === 1. Chargement des données ===
    print("\n📦 ÉTAPE 1 : Chargement des données")
    (cubes_train, cubes_eval, info_train, info_eval,
     zphot_train, zspec_eval, zphot_eval, survey_labels) = load_cosmos_data(
        CONFIG["DATA_DIR"], CONFIG["BAND_INDICES"]
    )

    # === 2. Normalisation par bande ===
    print("\n📐 ÉTAPE 2 : Normalisation par bande (z-score)")
    band_mean, band_std = compute_band_stats(cubes_train)
    print(f"   Moyennes par bande : {band_mean}")
    print(f"   Écarts-types       : {band_std}")

    cubes_train = normalize_cubes(cubes_train, band_mean, band_std)
    cubes_eval = normalize_cubes(cubes_eval, band_mean, band_std)

    # === 3. Augmentations & Dataset ===
    print("\n🎨 ÉTAPE 3 : Création des augmentations et du dataset")
    paired_transform = PairedGalaxyAugmentation(
        crop_size=CONFIG["CROP_SIZE"],
        noise_std_range=CONFIG["NOISE_STD_RANGE"],
        image_size=CONFIG["IMAGE_SIZE"]
    )
    dataset = BYOLGalaxyDataset(cubes_train, paired_transform)
    dataloader = DataLoader(
        dataset,
        batch_size=CONFIG["BATCH_SIZE"],
        shuffle=True,
        num_workers=CONFIG["NUM_WORKERS"],
        pin_memory=True,
        drop_last=True
    )

    # === 4. Modèle BYOL ===
    print("\n🧠 ÉTAPE 4 : Initialisation du modèle BYOL")
    model = BYOL(
        pretrained=CONFIG["PRETRAINED"],
        feat_dim=CONFIG["FEAT_DIM"],
        hidden_dim=CONFIG["HIDDEN_DIM"],
        proj_dim=CONFIG["PROJ_DIM"]
    )
    model = model.to(device)

    n_params = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Paramètres totaux    : {n_params:,}")
    print(f"   Paramètres entraînés : {n_trainable:,}")

    # === 5. Entraînement ===
    print("\n🚀 ÉTAPE 5 : Entraînement BYOL")
    history = train_byol(model, dataloader, CONFIG, device)

    # Courbes d'entraînement
    plot_training_curves(history, save_path=os.path.join(
        CONFIG["SAVE_DIR"], "training_curves.png"))

    # === 6. Extraction des features ===
    print("\n🔬 ÉTAPE 6 : Extraction des features")
    encoder = model.online_encoder
    encoder.eval()

    features_train = extract_features(encoder, cubes_train, device)
    features_eval = extract_features(encoder, cubes_eval, device)

    print(f"   Features entraînement : {features_train.shape}")
    print(f"   Features évaluation   : {features_eval.shape}")

    # Sauvegarde des features
    np.savez(os.path.join(CONFIG["SAVE_DIR"], "byol_features.npz"),
             features_train=features_train,
             features_eval=features_eval,
             zphot_train=zphot_train,
             zspec_eval=zspec_eval,
             zphot_eval=zphot_eval,
             survey_labels=survey_labels)
    print("   💾 Features sauvegardées dans byol_features.npz")

    # === 7. Évaluation ===
    print("\n📊 ÉTAPE 7 : Évaluation des représentations")

    # --- 7.1 t-SNE sur les données d'entraînement (coloré par ZPHOT) ---
    # Sous-échantillonnage si > 5000 pour t-SNE rapide
    max_tsne = 12000
    if len(features_train) > max_tsne:
        indices = np.random.choice(len(features_train), max_tsne, replace=False)
        feat_tsne = features_train[indices]
        z_tsne = zphot_train[indices]
        info_tsne = info_train[indices]
        print(f"   t-SNE : sous-échantillonnage {len(features_train)} → {max_tsne}")
    else:
        feat_tsne = features_train
        z_tsne = zphot_train
        info_tsne = info_train

    plot_tsne(feat_tsne, z_tsne,
              title="t-SNE des features BYOL — colorées par ZPHOT (entraînement)",
              save_path=os.path.join(CONFIG["SAVE_DIR"], "tsne_train_zphot.png"))

    # --- 7.2 t-SNE sur les données d'évaluation (coloré par ZSPEC) ---
    if len(features_eval) > 5:
        plot_tsne(features_eval, zspec_eval,
                  title="t-SNE des features BYOL — colorées par ZSPEC (évaluation)",
                  survey_labels=survey_labels,
                  save_path=os.path.join(CONFIG["SAVE_DIR"], "tsne_eval_zspec.png"))

    # --- 7.3 t-SNE combiné (train + eval) ---
    # Combiner les features pour voir la position des galaxies spec dans l'espace
    feat_combined = np.concatenate([feat_tsne, features_eval], axis=0)
    z_combined = np.concatenate([z_tsne, zphot_eval], axis=0)
    survey_combined = ["train"] * len(feat_tsne) + survey_labels

    plot_tsne(feat_combined, z_combined,
              title="t-SNE combiné — Train (ZPHOT) + Eval (ZSPEC)",
              survey_labels=survey_combined,
              save_path=os.path.join(CONFIG["SAVE_DIR"], "tsne_combined.png"))


    # --- 7.3.bis t-SNE multi-variantes ---
    print("\n🎨 Analyse t-SNE multi-variantes...")
    
    variants = [
        ('RA', 'RA (Ascension Droite)'),
        ('DEC', 'DEC (Déclinaison)'),
        ('i', 'Magnitude i'),
        ('EB_V', 'EBV (Extinction galactique)'),
        ('CLASS_STAR_HSC_I', 'CLASS_STAR (0=Galaxie, 1=Étoile)')
    ]
    
    for key, name in variants:
        if key in info_tsne.dtype.names:
            vals = info_tsne[key]
            # Pour la classe étoile, on gère les couleurs différemment si c'est binaire, mais plot_tsne gère les floats
            plot_tsne(feat_tsne, vals,
                      title=f"t-SNE BYOL — coloré par {name}",
                      save_path=os.path.join(CONFIG["SAVE_DIR"], f"tsne_train_{key}.png"))


    # --- 7.4 Similarité cosinus ---
    print("\n📐 Analyse de la similarité cosinus (données d'entraînement)")
    sim_matrix, bin_edges = compute_cosine_similarity_analysis(
        feat_tsne, z_tsne, n_bins=8
    )
    plot_cosine_similarity(sim_matrix, bin_edges,
                           save_path=os.path.join(CONFIG["SAVE_DIR"],
                                                  "cosine_similarity.png"))

    # === Rapport final ===
    print("\n" + "=" * 60)
    print("📋 RAPPORT FINAL")
    print("=" * 60)
    print(f"""
    🔭 Modèle : BYOL (Bootstrap Your Own Latent)
    🏗️ Encodeur : GalaxyEncoder (ResNet18 + stems par modalité)
       - Modalité optique : u, g, r (3 bandes)
       - Modalité red/NIR : i, z, y (3 bandes)
       - Poids ImageNet : {'Oui' if CONFIG['PRETRAINED'] else 'Non'}
    📦 Données : {len(cubes_train)} galaxies (entraînement) + {len(cubes_eval)} (évaluation)
    📐 Features : {CONFIG['FEAT_DIM']}-dim (après Global Average Pooling)
    🎯 Epochs : {CONFIG['NUM_EPOCHS']} | Batch : {CONFIG['BATCH_SIZE']}
    📉 Loss finale : {history['loss'][-1]:.4f}
    🎨 Augmentations : Random Crop (scale {CONFIG['CROP_SCALE']}) + Bruit Gaussien
    ⏱️ Temps total : {sum(history['lr'])*0 + (time.time() - 0):.0f}s

    📁 Fichiers générés :
       - checkpoints/byol_encoder_final.pth (encodeur entraîné)
       - checkpoints/byol_features.npz (features extraites)
       - checkpoints/training_curves.png
       - checkpoints/tsne_train_zphot.png
       - checkpoints/tsne_eval_zspec.png
       - checkpoints/tsne_combined.png
       - checkpoints/tsne_train_RA.png
       - checkpoints/tsne_train_DEC.png
       - checkpoints/tsne_train_i.png
       - checkpoints/tsne_train_EB_V.png
       - checkpoints/tsne_train_CLASS_STAR_HSC_I.png
       - checkpoints/cosine_similarity.png

    ⚠️ Limites :
       - Dataset modeste (~12K galaxies vs 1.3M pour ImageNet)
       - Seulement 27 galaxies avec ZSPEC pour l'évaluation quantitative
       - Augmentations minimales (crop + bruit uniquement)
       - Images 64×64 (petites pour un ResNet18 conçu pour 224×224)

    🔮 Suite logique :
       1. Fine-tuning supervisé : ajouter une tête de régression sur
          l'encodeur BYOL pour prédire le redshift
       2. Domain adaptation : appliquer les techniques adversariales
          de Treyer (2026) pour transférer D → UD
       3. Comparaison : features BYOL vs CNN supervisé vs SED-fitting
       4. Intégration de l'EBV (rougissement) comme feature additionnelle
          dans la couche FC du downstream task
    """)

    return model, features_train, features_eval, history


### Interprétation des variantes t-SNE

- **RA / DEC** : S'il n'y a pas de gradient visible, le modèle est invariant à la position spatiale (ce qui est recherché).
- **Magnitude i** : Un gradient indique que le modèle sépare les objets lumineux (haut SNR) des objets faibles, ce qui est normal et montre une sensibilité à la qualité du signal.
- **EBV** : L'absence de structure liée à l'EBV indique que l'encodeur est robuste à l'extinction galactique.
- **CLASS_STAR** : Permet de voir si les étoiles (CLASS_STAR proche de 1) sont isolées des galaxies.

In [ ]:
if __name__ == "__main__":
    model, features_train, features_eval, history = main()
